In [4]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\Vishakha
[nltk_data]     Singhal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Vishakha
[nltk_data]     Singhal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
df = pd.read_csv("Hotel_Reviews.csv")

print(df.head())
print(df.columns)

                                       Hotel_Address  \
0   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   
1   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   
2   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   
3   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   
4   s Gravesandestraat 55 Oost 1092 AA Amsterdam ...   

   Additional_Number_of_Scoring Review_Date  Average_Score   Hotel_Name  \
0                           194    8/3/2017            7.7  Hotel Arena   
1                           194    8/3/2017            7.7  Hotel Arena   
2                           194   7/31/2017            7.7  Hotel Arena   
3                           194   7/31/2017            7.7  Hotel Arena   
4                           194   7/24/2017            7.7  Hotel Arena   

  Reviewer_Nationality                                    Negative_Review  \
0              Russia    I am so angry that i made this post available...   
1             Ireland                                     

In [6]:
df = df[['Review_Total_Negative_Word_Counts',
         'Review_Total_Positive_Word_Counts',
         'Negative_Review',
         'Positive_Review',
         'Reviewer_Score']]

# Combine positive and negative reviews
df["Review"] = df["Positive_Review"] + " " + df["Negative_Review"]

# Create sentiment label
df["Sentiment"] = df["Reviewer_Score"].apply(lambda x: 1 if x >= 7 else 0)

df = df[['Review','Sentiment']]

print(df.head())

                                              Review  Sentiment
0   Only the park outside of the hotel was beauti...          0
1   No real complaints the hotel was great great ...          1
2   Location was good and staff were ok It is cut...          1
3   Great location in nice surroundings the bar a...          0
4   Amazing location and building Romantic settin...          0


In [8]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [lemmatizer.lemmatize(word)
             for word in words
             if word not in stop_words]

    return " ".join(words)

df["Review"] = df["Review"].apply(clean_text)

print(df.head())

                                              Review  Sentiment
0  park outside hotel beautiful angry made post a...          0
1  real complaint hotel great great location surr...          1
2  location good staff ok cute hotel breakfast ra...          1
3  great location nice surroundings bar restauran...          0
4  amazing location building romantic setting boo...          0


In [9]:
X = df["Review"]
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [11]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [12]:
y_pred = model.predict(X_test)

In [13]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8800267576685927

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.48      0.58     17497
           1       0.90      0.96      0.93     85651

    accuracy                           0.88    103148
   macro avg       0.81      0.72      0.75    103148
weighted avg       0.87      0.88      0.87    103148


Confusion Matrix:

[[ 8417  9080]
 [ 3295 82356]]


In [ ]:
while True:

    review = input("Enter a review: ")

    review = clean_text(review)

    review_vector = vectorizer.transform([review])

    prediction = model.predict(review_vector)[0]

    if prediction == 1:
        print("Positive Review 😊")
    else:
        print("Negative Review 😞")

    choice = input("Continue? (y/n): ")

    if choice.lower() != 'y':
        break

Enter a review:  garden outside was not bad


Positive Review 😊


Continue? (y/n):  y
Enter a review:  rooms were filthy


Negative Review 😞
